# B3 J1 · Atelier 1 · Spark sur de vraies données : 3 millions de courses de taxi

**Cours** : Data & IA B3 DEV · Sup de Vinci Nantes · vendredi 2 octobre 2026
**Format** : en binôme, **en autonomie**, 10h50-11h50 (pause 11h20-11h35 incluse)
**Livrable** : les 4 questions posées sur Qiplim à la fin du notebook (une réponse par binôme)

En B2, vous avez vu Spark sur un petit texte. Ici on passe à l'échelle : **un mois de courses de taxis jaunes à New York (janvier 2024, environ 3 millions de lignes, fichier Parquet de 50 Mo)**, publié par la ville de New York (NYC TLC).

## Ce que vous allez faire

| Heure | Étape |
|---|---|
| 10h50 | 0 · Setup (install + téléchargement), à lancer tout de suite |
| 11h00 | 1 · Lire du Parquet · 2 · Lazy evaluation |
| 11h10 | 3 · Spark SQL · 4 · Jointure + `.explain()` |
| 11h20 | **Pause** (retour 11h35) |
| 11h35 | 5 · Fonction de fenêtre · 6 · Écrire en Parquet · 7 · Le chrono Pandas / DuckDB / Spark |
| 11h45 | 8 · Les 4 questions sur Qiplim |

**Règles** : les cellules `# À VOUS` sont à compléter, la solution est juste en dessous (repliée) : essayez d'abord. Bloqués : 1. relisez l'erreur en entier, 2. demandez au binôme voisin, 3. ouvrez la solution. Personne ne reste bloqué seul plus de 10 minutes.

## 0 · Setup (≈ 1 minute)

Dans Colab : **Fichier > Enregistrer une copie dans Drive**, puis exécutez les deux cellules ci-dessous.
PySpark est installé en version 3.5 (compatible avec le Java de Colab).

In [ ]:
!pip install -q pyspark==3.5.3 duckdb setuptools

In [ ]:
import os, time, urllib.request
os.makedirs("data", exist_ok=True)
URLS = {
    "data/yellow_tripdata_2024-01.parquet": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet",
    "data/taxi_zone_lookup.csv": "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv",
}
for path, url in URLS.items():
    if not os.path.exists(path):
        urllib.request.urlretrieve(url, path)
    print(f"{path} : {os.path.getsize(path) / 1e6:.1f} Mo")

from pyspark.sql import SparkSession, functions as F, Window
spark = (SparkSession.builder
         .appName("B3-J1-taxis")
         .master("local[*]")
         .config("spark.sql.shuffle.partitions", "8")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "· cœurs disponibles :", spark.sparkContext.defaultParallelism)

## 1 · Lire du Parquet

Parquet est un format **en colonnes, compressé et typé** : le schéma est dans le fichier, et Spark peut ne lire que les colonnes utiles.

In [ ]:
t0 = time.time()
trips = spark.read.parquet("data/yellow_tripdata_2024-01.parquet")
print(f"read.parquet : {time.time() - t0:.2f} s  (rien n'est encore lu en entier : seul le schéma)")
trips.printSchema()

In [ ]:
t0 = time.time()
n = trips.count()   # une ACTION : là, Spark travaille
print(f"{n:,} courses · count() : {time.time() - t0:.2f} s · partitions : {trips.rdd.getNumPartitions()}")
trips.show(3, truncate=False)

## 2 · Lazy evaluation : rien ne tourne avant l'action

On construit une chaîne de nettoyage. Chronométrez : les transformations ne coûtent presque rien, c'est l'action qui paie.

In [ ]:
t0 = time.time()
clean = (trips
    .filter((F.col("trip_distance") > 0) & (F.col("fare_amount") > 0))
    .withColumn("duration_min", (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60)
    .filter(F.col("duration_min").between(1, 180))
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
    .withColumn("tip_pct", F.when(F.col("fare_amount") > 0, 100 * F.col("tip_amount") / F.col("fare_amount")))
)
t_transfo = time.time() - t0
print(f"5 transformations : {t_transfo:.3f} s")

t0 = time.time()
n_clean = clean.count()
t_action = time.time() - t0
print(f"count() après nettoyage : {n_clean:,} lignes · {t_action:.2f} s")
print(f"Lignes écartées : {n - n_clean:,} ({100 * (n - n_clean) / n:.1f} %)")

In [ ]:
# À VOUS
# Ajoutez à `clean` un filtre qui écarte les courses où passenger_count est nul (null) ou égal à 0,
# puis comptez les lignes restantes. Question : le temps de la transformation seule, et celui du count() ?

<details><summary>Solution (à ouvrir seulement après avoir essayé)</summary>

```python
clean2 = clean.filter(F.col("passenger_count").isNotNull() & (F.col("passenger_count") > 0))
t0 = time.time(); print(clean2.count(), f"{time.time() - t0:.2f} s")
```
</details>

## 3 · Spark SQL

Un DataFrame peut être interrogé en SQL : même moteur, même optimiseur (Catalyst), même plan.

In [ ]:
clean.createOrReplaceTempView("trips")
spark.sql('''
    SELECT pickup_hour,
           COUNT(*)                     AS courses,
           ROUND(AVG(fare_amount), 2)   AS prix_moyen,
           ROUND(AVG(tip_pct), 1)       AS pourboire_pct
    FROM trips
    GROUP BY pickup_hour
    ORDER BY pickup_hour
''').show(24)

In [ ]:
# À VOUS
# En SQL : les 5 modes de paiement (payment_type) avec leur nombre de courses et le pourboire moyen en %.
# Indice : payment_type 1 = carte, 2 = espèces. Que remarquez-vous sur les pourboires en espèces ?

<details><summary>Solution (à ouvrir seulement après avoir essayé)</summary>

```python
spark.sql('''
    SELECT payment_type, COUNT(*) AS courses, ROUND(AVG(tip_pct), 1) AS pourboire_pct
    FROM trips GROUP BY payment_type ORDER BY courses DESC LIMIT 5
''').show()
# Les pourboires en espèces valent ~0 % : ils ne sont pas saisis. Un biais de données classique.
```
</details>

## 4 · Jointure et lecture du plan avec `.explain()`

Les courses ne donnent qu'un identifiant de zone (`PULocationID`). La table des zones (265 lignes) donne le quartier (`Borough`) et le nom de la zone.

In [ ]:
zones = spark.read.csv("data/taxi_zone_lookup.csv", header=True, inferSchema=True)
zones.show(5)

par_quartier = (clean
    .join(zones, clean.PULocationID == zones.LocationID)
    .groupBy("Borough")
    .agg(F.count("*").alias("courses"),
         F.round(F.avg("fare_amount"), 2).alias("prix_moyen"),
         F.round(F.avg("trip_distance"), 2).alias("distance_moy"))
    .orderBy(F.desc("courses")))
par_quartier.show()

In [ ]:
par_quartier.explain()

**Lisez le plan de bas en haut** et cherchez :

- `FileScan parquet` : regardez `ReadSchema` (seules les colonnes utiles sont lues) et `PushedFilters` (les filtres envoyés directement au lecteur Parquet).
- `BroadcastHashJoin` : la table des zones est petite, Spark l'**envoie à chaque executor** au lieu de mélanger les 3 millions de lignes. Pas de shuffle pour la jointure.
- `Exchange hashpartitioning(Borough, ...)` : le **shuffle** du `groupBy`. Et `Exchange rangepartitioning` : celui du `orderBy`.
- `HashAggregate` deux fois : agrégat partiel sur chaque partition, puis agrégat final après le shuffle.

In [ ]:
par_quartier.explain(mode="formatted")   # même plan, présenté étape par étape

## 5 · Fonction de fenêtre : le top 3 des zones de départ par quartier

Une fenêtre calcule quelque chose **par groupe sans écraser les lignes** (classement, cumul, moyenne glissante).

In [ ]:
par_zone = (clean.join(zones, clean.PULocationID == zones.LocationID)
    .groupBy("Borough", "Zone").agg(F.count("*").alias("courses")))

w = Window.partitionBy("Borough").orderBy(F.desc("courses"))
top3 = (par_zone.withColumn("rang", F.row_number().over(w))
                .filter(F.col("rang") <= 3)
                .orderBy("Borough", "rang"))
top3.show(30, truncate=False)

In [ ]:
# À VOUS
# Même principe : pour chaque heure de départ (pickup_hour), la zone de départ la plus fréquente.
# Indice : groupBy("pickup_hour", "Zone"), puis une fenêtre partitionnée par pickup_hour.

<details><summary>Solution (à ouvrir seulement après avoir essayé)</summary>

```python
par_heure_zone = (clean.join(zones, clean.PULocationID == zones.LocationID)
    .groupBy("pickup_hour", "Zone").agg(F.count("*").alias("courses")))
w2 = Window.partitionBy("pickup_hour").orderBy(F.desc("courses"))
(par_heure_zone.withColumn("rang", F.row_number().over(w2))
    .filter("rang = 1").orderBy("pickup_hour").show(24, truncate=False))
```
</details>

## 6 · Écrire le résultat en Parquet partitionné

En data engineering, on n'affiche pas les résultats : on les **écrit** pour l'étape suivante (un modèle, un dashboard). Partitionner par une colonne crée un dossier par valeur : lire un seul quartier ne lira qu'un dossier.

In [ ]:
(clean.join(zones, clean.PULocationID == zones.LocationID)
      .select("tpep_pickup_datetime", "pickup_hour", "Borough", "Zone", "trip_distance", "fare_amount", "tip_pct", "duration_min")
      .write.mode("overwrite").partitionBy("Borough")
      .parquet("data/trips_clean"))
for d in sorted(os.listdir("data/trips_clean")):
    print(d)

In [ ]:
brooklyn = spark.read.parquet("data/trips_clean").filter(F.col("Borough") == "Brooklyn")
print(brooklyn.count(), "courses au départ de Brooklyn")
brooklyn.explain()   # cherchez PartitionFilters : Spark ne lit que le dossier Borough=Brooklyn

## 7 · Le chrono : Pandas, DuckDB, Spark sur la même question

Question : **prix moyen et nombre de courses par zone de départ, pour les courses de plus de 1 mile**. Notez les trois temps.

In [ ]:
import pandas as pd, duckdb

t0 = time.time()
pdf = pd.read_parquet("data/yellow_tripdata_2024-01.parquet")          # tout le fichier en RAM
res_pd = (pdf[pdf.trip_distance > 1].groupby("PULocationID")
          .agg(prix_moyen=("fare_amount", "mean"), courses=("fare_amount", "size")))
t_pandas = time.time() - t0
print(f"Pandas : {t_pandas:.2f} s · mémoire du DataFrame : {pdf.memory_usage(deep=True).sum() / 1e9:.2f} Go")
del pdf

t0 = time.time()
res_duck = duckdb.sql('''
    SELECT PULocationID, AVG(fare_amount) AS prix_moyen, COUNT(*) AS courses
    FROM 'data/yellow_tripdata_2024-01.parquet'
    WHERE trip_distance > 1 GROUP BY PULocationID
''').df()
t_duck = time.time() - t0
print(f"DuckDB : {t_duck:.2f} s")

t0 = time.time()
res_spark = (spark.read.parquet("data/yellow_tripdata_2024-01.parquet")
             .filter("trip_distance > 1").groupBy("PULocationID")
             .agg(F.avg("fare_amount").alias("prix_moyen"), F.count("*").alias("courses"))
             .collect())
t_spark = time.time() - t0
print(f"Spark  : {t_spark:.2f} s")
print("Mêmes résultats ?", len(res_pd) == len(res_duck) == len(res_spark))

**À discuter en binôme** : sur une seule machine et 3 millions de lignes, qui gagne ? Pourquoi Spark ne gagne-t-il pas ici (démarrage de tâches, planification, un seul ordinateur) ? Que se passerait-il avec **12 mois × 5 ans** de données (environ 180 millions de lignes, 15 Go en RAM pour Pandas) ?

## 8 · Les 4 questions à poster sur Qiplim (une réponse par binôme)

1. **Lazy evaluation** : combien de temps pour les 5 transformations de l'étape 2 ? Et pour le `count()` qui suit ? Expliquez l'écart.
2. **Lire le plan** : dans le plan de l'étape 4, quel nœud correspond au shuffle ? Pourquoi la jointure avec les zones n'en provoque-t-elle pas ?
3. **Le chrono** : classez Pandas, DuckDB et Spark sur l'étape 7, avec les temps. Pourquoi ce classement ?
4. **Le vrai monde** : à partir de quand Spark devient-il indispensable ? Donnez un exemple concret (volume, nombre de machines, type de données).

---

## Bonus si vous avez fini

- **Cache** : réutilisez `clean` trois fois (trois `count()`), chronométrez, puis `clean.cache()` et recommencez.
- **UDF Python contre fonction native** : calculez `tip_pct` avec une UDF Python (`F.udf`) et comparez le temps avec la version `F.col` ; regardez le plan (`BatchEvalPython`).
- **IA en copilote, avec vérification** : demandez à un assistant IA la requête « les 10 trajets zone de départ vers zone d'arrivée les plus rentables par minute ». Exécutez-la, puis **vérifiez** : le plan est-il raisonnable, les chiffres sont-ils plausibles (ordre de grandeur, nombre de lignes) ? Une requête qui tourne n'est pas une requête juste.

In [ ]:
# Spark UI (jobs, stages, DAG) : sur Colab, l'URL locale n'est pas accessible depuis votre navigateur ;
# en local (Jupyter sur votre machine), ouvrez http://localhost:4040 pendant qu'une action tourne.
print(spark.sparkContext.uiWebUrl)